## Milk Quality Prediction

### Setup

In [73]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score

### Data Preparation

In [85]:
# Load dataset
df_raw = pd.read_csv("../data/milknew.csv")
df = df_raw.copy()

# clean column names
df.columns = df.columns.str.lower().str.replace(" ", "_")
df.rename(columns={"temprature": "temperature"}, inplace=True)
df.rename(columns={"fat_": "fat"}, inplace=True)

# Grade column values mapping
grade_mapping = {"low": 0, "medium": 1, "high": 1}
df["grade"] = df["grade"].map(grade_mapping)

df.head()

,ph,temperature,taste,odor,fat,turbidity,colour,grade
0,6.6,35,1,0,1,0,254,1
1,6.6,36,0,1,0,1,253,1
2,8.5,70,1,1,1,1,246,0
3,9.5,34,1,1,0,1,255,0
4,6.6,37,0,0,0,0,255,1


In [86]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1059 entries, 0 to 1058
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   ph           1059 non-null   float64
 1   temperature  1059 non-null   int64  
 2   taste        1059 non-null   int64  
 3   odor         1059 non-null   int64  
 4   fat          1059 non-null   int64  
 5   turbidity    1059 non-null   int64  
 6   colour       1059 non-null   int64  
 7   grade        1059 non-null   int64  
dtypes: float64(1), int64(7)
memory usage: 66.3 KB


In [87]:
# Check missing values
df.isnull().sum()

ph             0
temperature    0
taste          0
odor           0
fat            0
turbidity      0
colour         0
grade          0
dtype: int64

In [88]:
# Validation framework
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train['grade'].values
y_val = df_val['grade'].values
y_test = df_test['grade'].values

del df_train['grade']
del df_val['grade']
del df_test['grade']

### EDA

#### Unique data

In [89]:

for col in df.columns:
    print(col)
    print(df[col].unique()[:5])
    print(df[col].nunique())
    print('-----')


ph
[6.6 8.5 9.5 5.5 4.5]
16
-----
temperature
[35 36 70 34 37]
17
-----
taste
[1 0]
2
-----
odor
[0 1]
2
-----
fat
[1 0]
2
-----
turbidity
[0 1]
2
-----
colour
[254 253 246 255 250]
9
-----
grade
[1 0]
2
-----


#### Data Balance

In [90]:
df_full_train['grade'].value_counts(normalize=True)

grade
1    0.602125
0    0.397875
Name: proportion, dtype: float64

In [91]:
# Mean
df_full_train['grade'].mean()

np.float64(0.602125147579693)

In [92]:
cols_numerical = ['ph', 'temperature', 'colour']
cols_categorical = ['taste', 'odor', 'fat', 'turbidity']

#### Categorical columns. Risk Ratio Analysis

In [93]:
# Risk Ratio Analysis
global_grade_mean = df_full_train['grade'].mean()
for c in cols_categorical:
    print(c)
    df_group = df_full_train.groupby(c).grade.agg(['mean', 'count'])
    df_group['diff'] = df_group['mean'] - global_grade_mean
    df_group['risk'] = df_group['mean'] / global_grade_mean
    display(df_group)
    print()

taste


,mean,count,diff,risk
taste,,,,
0,0.639098,399,0.036973,1.061404
1,0.569196,448,-0.032929,0.945312



odor


,mean,count,diff,risk
odor,,,,
0,0.626068,468,0.023943,1.039765
1,0.572559,379,-0.029566,0.950898



fat


,mean,count,diff,risk
fat,,,,
0,0.664286,280,0.062161,1.103235
1,0.571429,567,-0.030697,0.949020



turbidity


,mean,count,diff,risk
turbidity,,,,
0,0.786385,426,0.184260,1.306016
1,0.415677,421,-0.186448,0.690350


#### Categorical columns. Mutual Information

In [94]:
def mutual_info_grade_score(series):
    return mutual_info_score(series, df_full_train['grade'])

mi = df_full_train[cols_categorical].apply(mutual_info_grade_score)
mi.sort_values(ascending=False)

turbidity    0.073831
fat          0.004028
taste        0.002546
odor         0.001476
dtype: float64

#### Numerical columns. Correlation

In [ ]:
df_full_train[cols_numerical].corrwith(df_full_train['grade']).abs().sort_values(ascending=False)

temperature    0.499257
colour         0.177302
ph             0.015867
dtype: float64

### Training and evaluation

#### Logistic Regression

#### Decision Tree Classifier

#### Random Forests Classifier

#### Gradient Boosting Classifier

#### Model Selection

#### Save Model